# Chiffrage BAG BATTER SRL — marchés publics et devis client

Notebook de travail : il fait tourner l'outil du dépôt
[Devis-generator](https://github.com/pmeyssonnier/Devis-generator) et **range les
fichiers produits directement dans ton Drive**.

```
MyDrive/BAG_BATTER/Chiffrage/
    01_bibliotheque/     la bibliothèque de prix (6 onglets)
    02_metres_recus/     les métrés reçus des pouvoirs adjudicateurs
    03_offres_remises/   offres complétées et devis client, horodatés
    04_archives/         versions successives de la bibliothèque
```

**Ordre d'exécution** : les sections 1 et 2 d'abord (une fois par session),
puis n'importe quelle section 4 à 8, dans l'ordre qui t'arrange.

---

> ⚠️ **Les prix de la bibliothèque ne sont pas encore calibrés.** Les taux
> horaires et surtout les rendements (h/unité) sont des ordres de grandeur du
> marché belge, pas les chiffres de l'entreprise. Tant qu'ils n'ont pas été
> relus, ce notebook produit des documents qui montrent que la mécanique
> tourne — **ils ne sont pas prêts à partir chez un client ou une commune.**
> La section 4 est là pour cette relecture.

## 1. Installation

Récupère le code et la seule dépendance (openpyxl). À relancer à chaque
nouvelle session Colab : la machine est remise à zéro entre deux sessions.

Si le dépôt a changé depuis ton dernier passage, cette cellule le met à jour —
et dans ce cas fais **Exécution → Redémarrer la session** avant de continuer,
sinon Python garde en mémoire l'ancienne version du code.

In [ ]:
import os
import subprocess
import sys

REPO = '/content/Devis-generator'
URL  = 'https://github.com/pmeyssonnier/Devis-generator.git'

!pip install -q openpyxl

if os.path.isdir(os.path.join(REPO, '.git')):
    print(subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'],
                         capture_output=True, text=True).stdout.strip())
else:
    !git clone -q $URL $REPO
    print('Dépôt cloné.')

if REPO not in sys.path:
    sys.path.insert(0, REPO)

from chiffrage import bibliotheque as biblio
print(f"{len(biblio.RESSOURCES)} ressources · {len(biblio.OUVRAGES)} ouvrages "
      f"· {len(biblio.LOTS)} lots")

## 2. Drive et arborescence

Monte ton Drive (Colab demande une autorisation) et crée les quatre dossiers
s'ils n'existent pas encore. Sans risque à relancer : rien n'est écrasé.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/BAG_BATTER/Chiffrage')
DOSSIERS = {
    'bibliotheque': BASE / '01_bibliotheque',
    'metres':      BASE / '02_metres_recus',
    'offres':      BASE / '03_offres_remises',
    'archives':     BASE / '04_archives',
}
for chemin in DOSSIERS.values():
    chemin.mkdir(parents=True, exist_ok=True)
    print('✓', chemin)

## 3. Horodatage

Tout ce qui part chez un tiers est horodaté : deux versions d'une même offre
ne doivent jamais porter le même nom de fichier. On ne saurait plus laquelle a
été envoyée.

In [ ]:
from datetime import date, datetime

def horodate(nom, extension='.xlsx'):
    """'OFFRE_CSC-123' -> 'OFFRE_CSC-123_20260829_1430.xlsx'"""
    return f"{nom}_{datetime.now():%Y%m%d_%H%M}{extension}"

print(horodate('EXEMPLE'))

## 4. Contrôle et calibration

**À faire tourner avant toute utilisation sérieuse.**

`controle` vérifie l'intégrité des trois tables. `calibration` re-chiffre les
six devis forfaitaires historiques avec la bibliothèque et compare au montant
réellement vendu.

Comment lire les écarts :

- **écart négatif** — la bibliothèque chiffre moins cher que ce qui a été vendu ;
- **écart positif fort** — le chantier a été vendu sous son coût analytique,
  **ou** les quantités estimées sont trop élevées. Les deux hypothèses restent
  ouvertes tant que les surfaces réelles n'ont pas été relevées.

Cible : moins de 15 % d'écart **sur chaque ligne**, pas seulement en moyenne.

In [ ]:
from chiffrage.moteur import (controle_coherence, imprimer_calibration,
                              calcul_bordereau, coefficient_k)

anomalies = {k: v for k, v in controle_coherence().items() if v}
print('⚠️ ', anomalies) if anomalies else print('✅ Bibliothèque cohérente.')
print(f'Coefficient K = {coefficient_k():.4f}\n')

print(imprimer_calibration())

### Les 13 rendements les moins assis

Ces ouvrages ont été créés pour couvrir des postes qui restaient sans prix — ce
qui rendait toute offre irrégulière. Aucun n'a jamais été confronté à un
chantier réel : ce sont les premiers à valider.

In [ ]:
b = calcul_bordereau()
for code_ouv in biblio.OUVRAGES_A_VALIDER:
    ligne = b[code_ouv]
    print(f"{code_ouv}  {ligne['heures_mo']:>5.2f} h/{ligne['unite_ouv']:<4}"
          f"  {ligne['pu_vente']:>8.2f} €  {ligne['libelle_ouv']}")

## 5. Exporter la bibliothèque vers Drive

Écrit `01_bibliotheque/bibliotheque_prix_bagbatter.xlsx` (toujours le même nom :
c'est la version courante) **et** une copie horodatée dans `04_archives/`.

Le classeur porte de vraies formules : modifier un taux horaire dans l'onglet
RESSOURCES ou un coefficient dans PARAMS fait bouger tous les prix de vente.
C'est ce qui le rend utilisable sans Python.

⚠️ Les corrections faites **dans le classeur** ne remontent pas dans le code.
Le fichier sert à relire et à simuler ; une fois les bons chiffres connus, il
faut les reporter dans `chiffrage/bibliotheque.py` — sinon le prochain export
les écrase.

In [ ]:
from chiffrage.export_xlsx import exporter_bibliotheque

courante = DOSSIERS['bibliotheque'] / 'bibliotheque_prix_bagbatter.xlsx'
archive  = DOSSIERS['archives'] / horodate('bibliotheque_prix_bagbatter')

exporter_bibliotheque(str(courante))
exporter_bibliotheque(str(archive))
print('Version courante :', courante)
print('Archive          :', archive)

## 6. Devis client

Modifie le bloc `POSTES` : une ligne par ouvrage, `('code', quantité)`.
Les codes sont dans la section 9 (`bordereau`) ou dans l'onglet BORDEREAU du
classeur exporté.

**TVA** — 6 % uniquement pour un logement privé de plus de dix ans, à usage
principalement privé, facturé au consommateur final. Dans le doute, 21 %.

In [ ]:
from chiffrage.moteur import devis, imprimer_devis
from chiffrage.devis_xlsx import exporter_devis

# ── À MODIFIER ────────────────────────────────────────────────────
OBJET     = "Rénovation de la façade arrière"
REFERENCE = f"{date.today():%Y}-042"
CLIENT    = "M. et Mme Dupont\nRue de l'Église 12\n1030 Schaerbeek"
CHANTIER  = "Avenue Ernest Renan 62, 1030 Schaerbeek"
TVA       = 0.06          # 0.21 si les conditions du taux réduit ne sont pas remplies

POSTES = [
    ('10.10', 0.4),    # installation de chantier
    ('10.20', 24),     # échafaudage, m2
    ('20.10', 22),     # piquage de l'enduit dégradé, m2
    ('40.20', 22),     # enduit de façade armé, m2
    ('40.30', 22),     # peinture siloxane, m2
    ('20.50', 1.5),    # évacuation, m3
]
# ──────────────────────────────────────────────────────────────────

d = devis(OBJET, POSTES, tva=TVA)
print(imprimer_devis(d))

cible = DOSSIERS['offres'] / horodate(f'DEVIS_{REFERENCE}')
exporter_devis(d, str(cible), client=CLIENT, chantier=CHANTIER,
               reference=REFERENCE)
print('\nDevis écrit :', cible)

## 7. Répondre à un métré imposé

Le cas du marché public. Trois temps : déposer le métré reçu, le mapper, le
remplir.

### 7.1 Déposer le métré reçu

Exécute la cellule et choisis le fichier envoyé par le pouvoir adjudicateur ; il
est copié dans `02_metres_recus/`. Si tu l'as déjà déposé dans Drive à la main,
saute cette cellule — la suivante liste ce qui s'y trouve.

In [ ]:
from google.colab import files

for nom, contenu in files.upload().items():
    destination = DOSSIERS['metres'] / nom
    destination.write_bytes(contenu)
    print('Déposé :', destination)

### 7.2 Choisir le métré et vérifier la couverture

`MAPPING` fait le lien entre les codes du pouvoir adjudicateur (`NN.NN`) et nos
ouvrages (`LL.NN`). **Il est à refaire à chaque marché** : les codes
appartiennent au PA, pas à nous. Celui livré avec l'outil vaut pour le métré
d'entraînement (section 8).

Cette cellule ne modifie rien : elle dit seulement quels postes seraient
chiffrés et lesquels resteraient vides.

In [ ]:
from chiffrage.metre_io import lire_metre

INDEX = 0        # ← le numéro du métré à traiter, dans la liste ci-dessous

metres = sorted(DOSSIERS['metres'].glob('*.xlsx'))
for i, m in enumerate(metres):
    print(f'[{i}] {m.name}')

if not metres:
    raise SystemExit(
        f"Aucun métré dans {DOSSIERS['metres']}.\n"
        'Dépose-en un avec la cellule 7.1, ou génère le métré '
        "d'entraînement avec la section 8."
    )

METRE = metres[INDEX]

postes = lire_metre(str(METRE))
couverts = [p for p in postes if p['code'] in biblio.MAPPING]
orphelins = [p for p in postes if p['code'] not in biblio.MAPPING]

print(f"\n{METRE.name} — {len(postes)} postes, "
      f"{len(couverts)} couverts, {len(orphelins)} sans correspondance")
for p in orphelins:
    print(f"   {p['code']}  {p['designation'][:60]}  [{p['unite']}]")

### 7.3 Remplir l'offre

Écrit une **copie** du métré avec la colonne PU remplie : les quantités et les
formules du pouvoir adjudicateur restent intactes. C'est ce fichier que tu
renvoies.

Deux points de vigilance dans le rapport :

- **écarts d'unité** — un poste imposé au mètre courant face à un ouvrage au m²
  n'est **pas** chiffré. Aucune conversion automatique : c'est un arbitrage à
  faire à la main.
- **postes sans prix** — un seul suffit à rendre l'offre irrégulière et à la
  faire rejeter (art. 76 AR 18/04/2017). La liste doit être vide avant l'envoi.

In [ ]:
from chiffrage.metre_io import remplir_metre, imprimer_rapport

sortie = DOSSIERS['offres'] / horodate(f'OFFRE_{METRE.stem}')
rapport = remplir_metre(str(METRE), str(sortie), tva=0.21)
print(imprimer_rapport(rapport))

## 8. Métré d'entraînement

Un métré de marché public fictif (49 postes, 10 lots), pour essayer la
section 7 sans attendre un vrai cahier des charges. Il est déposé dans
`02_metres_recus/` comme s'il venait d'une commune.

In [ ]:
from chiffrage.gen_metre import generer_metre

cible = DOSSIERS['metres'] / 'METRE_CSC_2026-TP-0147_Schaerbeek.xlsx'
_, nb_postes, nb_lots = generer_metre(str(cible))
print(f'{nb_postes} postes · {nb_lots} lots -> {cible}')

## 9. Consulter les prix

`bordereau` liste les prix unitaires par lot. `fiche_prix` décompose un prix
ligne par ligne : c'est la pièce à produire si un pouvoir adjudicateur demande
la justification d'un prix jugé anormal (art. 36 AR 18/04/2017).

In [ ]:
bordereau = calcul_bordereau()

for code_ouv in sorted(bordereau):
    ligne = bordereau[code_ouv]
    print(f"{code_ouv}  {ligne['pu_vente']:>9.2f} €/{ligne['unite_ouv']:<5}"
          f" {ligne['heures_mo']:>6.3f} h  {ligne['libelle_ouv']}")

In [ ]:
from chiffrage.moteur import fiche_prix

print(fiche_prix('40.20'))      # ← le code de l'ouvrage à justifier